# Heart attack risk — walkthrough notebookThis notebook is for **understanding**, not for submission. Your actual submission is`train.py` + `app.py` in the repo. This is the Colab-style version so you can run thingsone cell at a time and see what each step does.**Run every cell in order, top to bottom.** In Colab: `Runtime -> Run all`, or Shift+Enterone at a time.### What we're buildingA model that looks at someone's survey answers and estimates the probability they have hada heart attack. We build it **three different ways** that should all give the same answer:1. **scikit-learn** — one line, `LogisticRegression().fit(X, y)`2. **Manual PyTorch** — we write the gradient descent loop ourselves3. **PyTorch `nn.Module`** — the standard PyTorch wayWhy three? Because the assignment is testing whether you understand what `.fit()` isactually doing. If all three agree, you have proved you know what is inside the black box.

## Part 0 — Setup

In [ ]:
# Works in Colab and locally. In Colab this installs what's missing (~30 seconds).
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q scikit-learn torch pandas matplotlib

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print("Colab:", IN_COLAB)
print("pandas", pd.__version__, "| torch", torch.__version__)

In [ ]:
# The 20,000-row sample lives in the public GitHub repo, so Colab can just download it.
# (Locally, it reads the file already in your repo.)
import os

URL = "https://raw.githubusercontent.com/Wilrick1234/daiist-assignment-1/main/data/heart_2022_sample.csv"
LOCAL = "data/heart_2022_sample.csv"

raw = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL)
print(f"Loaded {len(raw):,} rows x {raw.shape[1]} columns")

## Part 1 — EDA: look at the data before touching a modelThis part is not busywork. **Feature engineering is 30% of your grade**, and you cannotengineer a feature you have not looked at. Every chart below is a potential paragraph inREPORT.md.

In [ ]:
# What does one row look like? One row = one adult who answered the CDC phone survey.
raw.head(3)

In [ ]:
# All the columns we have to work with.
for i, c in enumerate(raw.columns):
    print(f"{i:2d}. {c:28s} {str(raw[c].dtype):8s}  {raw[c].nunique():>5} unique")

### The single most important fact about this dataset

In [ ]:
counts = raw["HadHeartAttack"].value_counts(dropna=False)
rate = raw["HadHeartAttack"].eq("Yes").mean()

print(counts, "\n")
print(f"Share who had a heart attack: {rate:.2%}")
print()
print("Now the trap:")
print(f"  A model that just says 'No' to everyone is {1-rate:.2%} accurate.")
print(f"  It also catches ZERO heart attacks. It is completely useless.")
print()
print("This is why we never use accuracy here. We use PR-AUC and recall instead.")

**Write that down.** "Accuracy is the wrong metric because the classes are imbalanced" isexactly the kind of thing the comprehension check asks, and the chart below is why.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(["Did NOT have one", "Had a heart attack"],
        [1 - rate, rate], color=["#bbb", "#c0392b"])
ax.set_xlim(0, 1); ax.set_xlabel("share of people")
for i, v in enumerate([1 - rate, rate]):
    ax.text(v + 0.01, i, f"{v:.1%}", va="center")
ax.set_title("Class imbalance: this is why accuracy lies")
plt.tight_layout(); plt.show()

### Accuracy vs recall — the difference, concretelyThese two measure different things, and mixing them up is the classic mistake withimbalanced data.- **Accuracy** = of *everyone*, what share did I label correctly? It counts the 18,864  healthy people and the 1,136 heart attack cases equally.- **Recall** = of the *heart attack cases only*, what share did I catch? Healthy people are  not in this calculation at all.Accuracy can be dragged up to 94% just by being right about the easy majority. Recall cannotbe faked — it only ever asks about the people you actually care about.

In [ ]:
n_pos, n_neg = int(y_bin_total := raw["HadHeartAttack"].eq("Yes").sum()), int((~raw["HadHeartAttack"].eq("Yes")).sum())
N = n_pos + n_neg

def score(tp, fn, fp, tn, name):
    acc = (tp + tn) / N
    rec = tp / (tp + fn) if (tp + fn) else 0
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    print(f"{name}")
    print(f"   caught {tp:5d} of {n_pos} cases | missed {fn:5d} | false alarms {fp:6d}")
    print(f"   ACCURACY = {acc:6.2%}     RECALL = {rec:6.2%}     precision = {prec:.2%}" if tp else
          f"   ACCURACY = {acc:6.2%}     RECALL = {rec:6.2%}     precision = n/a")
    print()

score(0, n_pos, 0, n_neg, "A) Say 'No' to everyone  <- the lazy model")
score(n_pos, 0, n_neg, 0, "B) Flag everyone         <- the paranoid model")

print("Model A looks excellent on accuracy and is worthless: it misses all 1,136 cases.")
print("Model B has perfect recall and is also useless: it alarms on 18,864 healthy people.")
print()
print("Neither extreme is the goal. You want a model that trades these off deliberately,")
print("and the threshold slider is where you make that trade.")

**The one-line version for your comprehension check:** accuracy asks "how often am I right?",recall asks "how many of the real cases did I catch?" When only 5.68% of people are positive,you can be right 94% of the time while catching nobody — so accuracy is the wrong thing tooptimize, and recall (together with precision, summarized by PR-AUC) is the right one.There is always a trade: pushing recall up means flagging more people, which drags precisiondown. That trade is exactly what `COST_FN` and `COST_FP` are for — they let you say *howmuch* a miss hurts compared to a false alarm, so the maths can pick the balance for you.

### Missing values

In [ ]:
miss = raw.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print(f"{len(miss)} of {raw.shape[1]} columns have missing values.\n")
print((miss * 100).round(1).to_string())

We are using the **with NaNs** version of the file on purpose. The pipeline fills these in(median for numbers, most-common for text). But *how* you fill them is a real featureengineering decision — e.g. is "did not answer the HIV testing question" itself informative?That is a defensible feature you could add.

### The key EDA move: heart attack rate per categoryFor every text column, we ask: *does the heart attack rate change a lot between itscategories?* A column where it swings wildly is a strong predictor. A column where everycategory sits at ~5.7% is useless.

In [ ]:
def rate_by(col, min_n=50):
    # Heart attack rate within each category of `col`, plus how many people are in it.
    g = raw.groupby(col, dropna=False)["HadHeartAttack"]
    out = pd.DataFrame({"n": g.size(), "rate": g.apply(lambda s: s.eq("Yes").mean())})
    # People who didn't answer become their own visible category rather than vanishing —
    # "declined to answer" can itself be informative.
    out.index = pd.Index([("(no answer)" if pd.isna(v) else str(v)) for v in out.index], name=col)
    return out[out["n"] >= min_n].sort_values("rate", ascending=False)

def plot_rate(col):
    d = rate_by(col)
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(d) + 1.2))
    ax.barh(list(d.index)[::-1], list(d["rate"])[::-1], color="#2c7fb8")
    ax.axvline(rate, color="#c0392b", ls="--", label=f"overall {rate:.1%}")
    ax.set_xlabel("share who had a heart attack"); ax.legend()
    ax.set_title(f"{col}")
    plt.tight_layout(); plt.show()
    return d

plot_rate("GeneralHealth")

Look at that spread: people who rate their health "Poor" have a far higher rate than"Excellent". And notice the **order is meaningful** — Excellent < Very good < Good < Fair <Poor. Right now the pipeline one-hot encodes this, which throws the ordering away. Turningit into a number 0–4 instead is a feature engineering idea worth testing.

In [ ]:
plot_rate("AgeCategory")

Same story: age is clearly ordered and clearly predictive. Another one-hot-vs-number candidate.

### The leakage question — this one is yours to decideSome columns might tell the model the answer in disguise.

In [ ]:
for col in ["HadAngina", "ChestScan", "HadStroke", "HadDiabetes", "DifficultyWalking"]:
    d = rate_by(col)
    spread = d["rate"].max() - d["rate"].min()
    print(f"\n{col}  (spread {spread:.1%})")
    print(d.assign(rate=lambda x: (x['rate'] * 100).round(1)).to_string())

**Think about `HadAngina` and `ChestScan`.**Angina is chest pain from the same underlying coronary disease that causes heart attacks.A chest scan is often ordered *because* someone already had a cardiac event.So: is a model that uses these actually predicting anything, or is it just reading themedical paper trail of an event that already happened? The answer depends entirely on **whoyou say is using the model** — which is why the business framing has to come first.Whatever you decide goes in `DROP_COLUMNS` in `common.py`. There is no single right answer,but there is a wrong one: not thinking about it.### Measure what each suspect is actually worthArguing about leakage in the abstract is weak. Measuring it is strong. The cell belowretrains the model with different columns removed, so you can say exactly what each onecosts you.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split as _tts
from sklearn.compose import ColumnTransformer as _CT
from sklearn.impute import SimpleImputer as _SI
from sklearn.pipeline import Pipeline as _P
from sklearn.preprocessing import OneHotEncoder as _OHE, StandardScaler as _SS
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.metrics import average_precision_score as _ap, roc_auc_score as _roc

def leakage_test(drop_cols, label):
    df = raw.drop(columns=drop_cols)
    yy = df["HadHeartAttack"].eq("Yes").astype(int).to_numpy()
    XX = df.drop(columns=["HadHeartAttack"])
    Xtv, _, ytv, _ = _tts(XX, yy, test_size=.2, stratify=yy, random_state=42)
    Xtr, Xva, ytr, yva = _tts(Xtv, ytv, test_size=.25, stratify=ytv, random_state=42)
    num = [c for c in Xtr.columns if pd.api.types.is_numeric_dtype(Xtr[c])]
    cat = [c for c in Xtr.columns if c not in num]
    pipe = _P([("e", _CT([("n", _SI(strategy="median"), num),
                          ("c", _P([("i", _SI(strategy="most_frequent")),
                                    ("o", _OHE(drop="first", handle_unknown="ignore",
                                               sparse_output=False))]), cat)],
                         verbose_feature_names_out=False)), ("s", _SS())])
    A, B = pipe.fit_transform(Xtr), pipe.transform(Xva)
    m = _LR(C=0.001, max_iter=10_000).fit(A, ytr)
    p = m.predict_proba(B)[:, 1]
    print(f"{label:38s} PR-AUC={_ap(yva, p):.4f}   ROC-AUC={_roc(yva, p):.4f}")
    return _ap(yva, p)

base_pr = leakage_test([], "keep everything")
leakage_test(["HadAngina"], "drop HadAngina")
leakage_test(["ChestScan"], "drop ChestScan")
leakage_test(["HadAngina", "ChestScan"], "drop both")
leakage_test(["HadAngina", "ChestScan", "HadStroke"], "drop both + HadStroke")

Read those numbers carefully, because they tell two completely different stories:- **`HadAngina` is load-bearing.** Removing it collapses PR-AUC by roughly 38%. Almost all  the model's apparent skill is coming from this one column.- **`ChestScan` is nearly free to remove.** It barely changes anything, because whatever it  knows is already covered by the other health columns.So the honest framing is: *this model is mostly an angina detector.* Whether that is aproblem depends on your scenario:| If your scenario is... | Then `HadAngina` is... ||---|---|| A nurse with the patient's full chart in front of them | Fine — they genuinely know this || A public-health screening questionnaire sent by post | Fine — it is just another survey question || An insurer scoring someone before any medical contact | Questionable — would they really have this? || "Predicting who *will* have a heart attack" | A serious problem — it is part of the same disease |Pick a lane, say so in REPORT.md, and back it with the numbers above. **Showing you measuredthis and chose deliberately is worth far more marks than a higher PR-AUC.**

### Which columns look most promising overall? ("spread")**Spread** is just a number I am computing to rank columns. For one column, look at theheart attack rate inside each of its categories, then take:> **spread = highest category rate − lowest category rate**Example with `GeneralHealth`: people rating their health "Poor" have a much higher rate thanpeople rating it "Excellent". Say those are 18% and 1.5% — the spread is 16.5 percentagepoints. That column clearly separates people.Now imagine a column where every category sits at about 5.7% (the overall rate). Highestminus lowest is near zero. That column tells you nothing; knowing someone's answer does notchange your estimate at all.So: **big spread = the column carries signal. Near-zero spread = it is probably noise.**It is a rough-and-ready measure — it ignores how many people are in each category, which iswhy the cell filters out groups smaller than 50. The Cramér's V further down is the morerigorous version of the same idea.

In [ ]:
cat_cols = [c for c in raw.columns
            if not pd.api.types.is_numeric_dtype(raw[c]) and c != "HadHeartAttack"]

rows = []
for c in cat_cols:
    d = rate_by(c)
    if len(d) >= 2:
        rows.append({"column": c, "n_categories": len(d),
                     "min_rate": d["rate"].min(), "max_rate": d["rate"].max(),
                     "spread": d["rate"].max() - d["rate"].min()})

ranked = pd.DataFrame(rows).sort_values("spread", ascending=False).reset_index(drop=True)
ranked[["min_rate", "max_rate", "spread"]] = (ranked[["min_rate", "max_rate", "spread"]] * 100).round(1)
ranked

This table is your feature engineering shortlist. Big spread at the top = worth yourattention. Near-zero spread at the bottom = candidates for dropping (fewer useless columnsmeans less noise for the model to overfit).

### Correlation: what is related to what?Two different questions here, and they need different tools.**Question 1: which columns relate to the target?** For numeric columns that is ordinarycorrelation. For text columns, correlation is undefined — you cannot correlate "Excellent"with anything — so we use **Cramér's V** instead, which measures association between twocategorical variables on the same 0-to-1 scale. 0 means no relationship, 1 means oneperfectly determines the other.

In [ ]:
from scipy.stats import chi2_contingency

y_bin = raw[TARGET_COL].eq("Yes").astype(int) if (TARGET_COL := "HadHeartAttack") else None

def cramers_v(col):
    tab = pd.crosstab(raw[col].fillna("(missing)"), y_bin)
    if tab.shape[0] < 2:
        return np.nan
    chi2 = chi2_contingency(tab)[0]
    phi2 = chi2 / tab.values.sum()
    r, k = tab.shape
    return np.sqrt(phi2 / max(1e-12, min(k - 1, r - 1)))

rows = []
for c in raw.columns:
    if c == "HadHeartAttack":
        continue
    if pd.api.types.is_numeric_dtype(raw[c]):
        rows.append({"column": c, "kind": "numeric", "strength": abs(raw[c].corr(y_bin))})
    else:
        rows.append({"column": c, "kind": "categorical", "strength": cramers_v(c)})

assoc = pd.DataFrame(rows).dropna().sort_values("strength", ascending=False).reset_index(drop=True)
assoc.head(20)

In [ ]:
top20 = assoc.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top20["column"], top20["strength"],
        color=["#c0392b" if c in ("HadAngina", "ChestScan") else "#2c7fb8"
               for c in top20["column"]])
ax.set_xlabel("association with having had a heart attack")
ax.set_title("Strongest columns (red = the ones you need to think hard about)")
plt.tight_layout(); plt.show()

Notice how far `HadAngina` sticks out — more than twice the next column. **A single featurethat dominates everything else is a red flag, not a win.** It usually means the feature istoo close to the thing you are predicting. Hold that thought for the leakage section below.**Question 2: which columns duplicate each other?** If two features carry the sameinformation, you are feeding the model the same thing twice. For a linear model that makescoefficients unstable and hard to interpret.

In [ ]:
num_cols_all = [c for c in raw.columns if pd.api.types.is_numeric_dtype(raw[c])]
corr = raw[num_cols_all].corr()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols_all))); ax.set_xticklabels(num_cols_all, rotation=45, ha="right")
ax.set_yticks(range(len(num_cols_all))); ax.set_yticklabels(num_cols_all)
for i in range(len(num_cols_all)):
    for j in range(len(num_cols_all)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Numeric columns vs each other")
plt.tight_layout(); plt.show()

Look for the dark squares off the diagonal. **BMI, WeightInKilograms and HeightInMeters**should light up, because BMI *is* weight / height². That is three columns holding twocolumns' worth of information. Dropping one is a defensible feature engineering move — andnow you have a chart to justify it with.

In [ ]:
# Text columns can be redundant too. The six "Difficulty / disability" questions:
dis = ["DifficultyWalking", "DifficultyErrands", "DifficultyDressingBathing",
       "DifficultyConcentrating", "DeafOrHardOfHearing", "BlindOrVisionDifficulty"]

dis_bin = (raw[dis] == "Yes").astype(int)
dcorr = dis_bin.corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dcorr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(dis))); ax.set_xticklabels(dis, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(dis))); ax.set_yticklabels(dis, fontsize=8)
for i in range(len(dis)):
    for j in range(len(dis)):
        ax.text(j, i, f"{dcorr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(dcorr.iloc[i, j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

print("These six all point the same direction. Rather than six separate one-hot columns,")
print("you could collapse them into a single 'how many difficulties does this person have'")
print("count (0-6). Fewer columns, same information, and an ordered scale the model can use.")

### Numeric columns

In [ ]:
num_cols = [c for c in raw.columns if pd.api.types.is_numeric_dtype(raw[c])]
print("Numeric columns:", num_cols, "\n")

# Compare the average value for people who did vs didn't have a heart attack.
comp = raw.groupby(raw["HadHeartAttack"].eq("Yes"))[num_cols].mean().T
comp.columns = ["No heart attack", "Had heart attack"]
comp["difference"] = comp["Had heart attack"] - comp["No heart attack"]
comp.round(2)

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(3.2 * len(num_cols), 3))
for ax, c in zip(np.atleast_1d(axes), num_cols):
    for val, lab, col in [(False, "No", "#bbb"), (True, "Yes", "#c0392b")]:
        ax.hist(raw.loc[raw["HadHeartAttack"].eq("Yes") == val, c].dropna(),
                bins=30, density=True, alpha=0.55, label=lab, color=col)
    ax.set_title(c, fontsize=9); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# BMI is literally weight / height^2 — so these three columns overlap heavily.
check = raw[["BMI", "WeightInKilograms", "HeightInMeters"]].dropna()
recomputed = check["WeightInKilograms"] / check["HeightInMeters"] ** 2
print("Correlation between stored BMI and recomputed BMI:",
      round(np.corrcoef(check["BMI"], recomputed)[0, 1], 6))
print("\nDo you need all three columns, or does keeping them just duplicate information?")
print("That is a feature engineering decision you can defend either way.")

## Part 2 — The modeling, step by stepNow we build the model. Each cell below mirrors a piece of `train.py`, broken apart so youcan see it happen.

In [ ]:
TARGET = "HadHeartAttack"
SEED = 42

# y = 1 if they had a heart attack, 0 otherwise. X = everything else.
y_all = raw[TARGET].eq("Yes").astype(int).to_numpy()
X_all = raw.drop(columns=[TARGET])

print("X:", X_all.shape, " y:", y_all.shape, " positives:", y_all.sum())

### Splitting the data — 60% train / 20% validation / 20% testThree parts, each with a different job:| Part | Job ||---|---|| **Train** (60%) | The model learns its coefficients from these rows || **Validation** (20%) | We pick settings here — regularization strength, decision threshold || **Test** (20%) | Touched **once**, at the very end, for honest final numbers |`stratify=` keeps the 5.7% positive rate identical in all three. Without it, a random splitcould easily give one part noticeably more heart attacks than another.**There are no dates in this survey**, so a random split cannot leak the future into thepast. If this were, say, transaction data, we would have to split by time instead. Say thisin REPORT.md — the README asks for it specifically.

In [ ]:
from sklearn.model_selection import train_test_split

X_tv, X_test, y_tv, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=SEED)

for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s} {len(yy):6,} rows   positive rate {yy.mean():.2%}")

### PreprocessingThree things happen here:1. **Impute** — fill missing values (median for numbers, most common for text)2. **One-hot encode** — a model cannot multiply the word "Excellent". So `GeneralHealth`   becomes several 0/1 columns, one per category.3. **Standardize** — rescale everything to mean 0, std 1.Step 3 matters more than it looks. Gradient descent struggles when one column ranges 0–300and another 0–1. Standardizing is what lets our hand-written PyTorch loop actually convergeto the same answer sklearn gets.**Critical:** we `fit` on the training rows only, then `transform` all three. Fitting oneverything would let test-set information bleed into training.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
categorical = [c for c in X_train.columns if c not in numeric]

pre = Pipeline([
    ("encode", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore",
                                     sparse_output=False)),
        ]), categorical),
    ], verbose_feature_names_out=False)),
    ("scale", StandardScaler()),
])

Xtr = pre.fit_transform(X_train)      # fit ONLY on train
Xva = pre.transform(X_val)            # then just transform
Xte = pre.transform(X_test)

feat_names = pre.named_steps["encode"].get_feature_names_out()
print(f"Before: {X_train.shape[1]} columns ({len(numeric)} numeric + {len(categorical)} text)")
print(f"After:  {Xtr.shape[1]} columns  <- one-hot expanded the text ones")

In [ ]:
# See the expansion concretely: what did GeneralHealth turn into?
print("GeneralHealth became these columns:")
for f in feat_names:
    if f.startswith("GeneralHealth"):
        print("   ", f)
print("\n'Excellent' is missing on purpose — that's drop='first'. It becomes the reference")
print("category that the intercept absorbs, which avoids perfect collinearity.")

### Model 1 of 3 — scikit-learnThe one-liner. `C` controls regularization: **smaller C = stronger penalty on largecoefficients = simpler model**. This is the one hyperparameter worth tuning, and we tune iton validation.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    m = LogisticRegression(C=C, max_iter=10_000, tol=1e-10).fit(Xtr, y_train)
    pr = average_precision_score(y_val, m.predict_proba(Xva)[:, 1])
    print(f"C={C:<8} validation PR-AUC = {pr:.4f}")

In [ ]:
C = 0.01        # the winner above
sk = LogisticRegression(C=C, max_iter=10_000, tol=1e-10).fit(Xtr, y_train)
print("Done. Learned", len(sk.coef_[0]), "coefficients + 1 intercept.")

### Model 2 of 3 — the manual PyTorch loopThis is the cell to actually read. It is what `sk.fit()` above does internally, written out.The model is: `probability = sigmoid(X @ w + b)`. We start with all weights at zero andrepeatedly:1. **Forward** — compute predictions with the current weights2. **Loss** — measure how wrong they are (binary cross-entropy + the regularization penalty)3. **Backward** — `loss.backward()` computes the gradient (the direction that makes loss worse)4. **Update** — step the weights in the *opposite* direction3,000 times. That's it. That's gradient descent.**The one conversion you must be able to explain:** sklearn writes its objective as`0.5*||w||² + C * Σ(errors)`. We write ours as `mean(errors) + (λ/2)*||w||²`. Dividingsklearn's by `C*n` makes them identical when **λ = 1/(C·n)**. Get this wrong and the threemodels will not agree.

In [ ]:
Xt = torch.tensor(Xtr, dtype=torch.float64)
yt = torch.tensor(y_train, dtype=torch.float64)
n, d = Xt.shape

lam = 1.0 / (C * n)       # <- the conversion
LR, EPOCHS = 0.5, 3000

w = torch.zeros(d, dtype=torch.float64, requires_grad=True)
b = torch.zeros(1, dtype=torch.float64, requires_grad=True)
history = []

for epoch in range(EPOCHS):
    z = Xt @ w + b                                          # 1. forward: logits
    p = (1 / (1 + torch.exp(-z))).clamp(1e-12, 1 - 1e-12)   #    sigmoid -> probability
    bce = -(yt * torch.log(p) + (1 - yt) * torch.log(1 - p)).mean()
    loss = bce + 0.5 * lam * (w ** 2).sum()                 # 2. loss (+ penalty)

    loss.backward()                                          # 3. backward: fill .grad
    with torch.no_grad():                                    # 4. update, untracked
        w -= LR * w.grad
        b -= LR * b.grad
    w.grad.zero_(); b.grad.zero_()                           #    reset, or they accumulate

    history.append(loss.item())
    if epoch % 500 == 0:
        print(f"epoch {epoch:5d}   loss {loss.item():.6f}")

print(f"epoch {EPOCHS:5d}   loss {history[-1]:.6f}   <- converged")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history, color="#2c7fb8")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_yscale("log")
ax.set_title("Gradient descent: loss going down, then flattening out")
plt.tight_layout(); plt.show()

### Model 3 of 3 — standard PyTorch (`nn.Module` + `optim`)Same maths, idiomatic API. `nn.Linear` holds `w` and `b` for us, `BCEWithLogitsLoss` is thesigmoid+cross-entropy fused (and numerically safer), and `optimizer.step()` does the update.Two details that make this match the manual loop exactly:- We **zero-initialize** the weights, because `nn.Linear` normally starts random.- `weight_decay=lam` adds `lam*w` to the gradient — which is exactly the derivative of  `(lam/2)*||w||²`. We apply it to the **weights only, not the bias**, matching sklearn.  Penalizing the bias would shift the model's base rate for no good reason.

In [ ]:
from torch import nn

model = nn.Linear(d, 1).to(torch.float64)
nn.init.zeros_(model.weight); nn.init.zeros_(model.bias)

opt = torch.optim.SGD(
    [{"params": [model.weight], "weight_decay": lam},
     {"params": [model.bias],   "weight_decay": 0.0}],
    lr=LR)
loss_fn = nn.BCEWithLogitsLoss()

hist2 = []
for epoch in range(EPOCHS):
    opt.zero_grad()
    loss = loss_fn(model(Xt).squeeze(1), yt)
    loss.backward()
    opt.step()
    with torch.no_grad():
        hist2.append(loss.item() + 0.5 * lam * model.weight.pow(2).sum().item())

print(f"final loss {hist2[-1]:.6f}")

### Do the three agree?

In [ ]:
def probs_manual(Xmat):
    return 1 / (1 + np.exp(-(Xmat @ w.detach().numpy() + b.detach().numpy())))

def probs_module(Xmat):
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(Xmat, dtype=torch.float64)).squeeze(1)).numpy()

p_sk, p_mn, p_md = sk.predict_proba(Xte)[:, 1], probs_manual(Xte), probs_module(Xte)

print("Biggest difference in predicted probability, across all 4,000 test people:")
print(f"  sklearn vs manual PyTorch : {np.abs(p_sk - p_mn).max():.2e}")
print(f"  sklearn vs nn.Module      : {np.abs(p_sk - p_md).max():.2e}")
print(f"  manual vs nn.Module       : {np.abs(p_mn - p_md).max():.2e}")
print()
print("Biggest difference in the learned coefficients:")
print(f"  sklearn vs manual PyTorch : {np.abs(sk.coef_[0] - w.detach().numpy()).max():.2e}")
print(f"  manual vs nn.Module       : {np.abs(w.detach().numpy() - model.weight.detach().numpy()[0]).max():.2e}")

Those numbers should be tiny. **That is the whole point of the assignment.** Three verydifferent-looking pieces of code, one identical model, because they minimize the sameobjective and all reached the bottom.Now break it on purpose, so you understand *why* it worked:

In [ ]:
# Run far too few epochs and watch the agreement fall apart.
for test_epochs in [50, 200, 1000, 3000]:
    w2 = torch.zeros(d, dtype=torch.float64, requires_grad=True)
    b2 = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    for _ in range(test_epochs):
        z = Xt @ w2 + b2
        p = (1 / (1 + torch.exp(-z))).clamp(1e-12, 1 - 1e-12)
        l = -(yt * torch.log(p) + (1 - yt) * torch.log(1 - p)).mean() + 0.5 * lam * (w2 ** 2).sum()
        l.backward()
        with torch.no_grad():
            w2 -= LR * w2.grad; b2 -= LR * b2.grad
        w2.grad.zero_(); b2.grad.zero_()
    gap = np.abs(sk.coef_[0] - w2.detach().numpy()).max()
    print(f"{test_epochs:5d} epochs -> max coefficient gap vs sklearn: {gap:.2e}")

There is your comparison section for REPORT.md: they agree **because** they converged, andyou can show exactly what happens when they don't. Try the same with `LR = 0.01`.

### Evaluation — and picking the decision thresholdThe model outputs a probability. To actually *do* something you need a cutoff: flag peopleabove it. Where you put that cutoff is a **business decision**, not a statistical one.We pick it on **validation**, then report on **test**. Picking it on test would meanreporting a number we tuned to, which is cheating.

In [ ]:
COST_FN = 10.0   # cost of missing a real case      <- YOUR number to justify
COST_FP = 1.0    # cost of a false alarm            <- YOUR number to justify

p_val = sk.predict_proba(Xva)[:, 1]

def cost_at(thresh, yy, pp):
    flag = (pp >= thresh).astype(int)
    fn = ((flag == 0) & (yy == 1)).sum()
    fp = ((flag == 1) & (yy == 0)).sum()
    return COST_FN * fn + COST_FP * fp

grid = np.linspace(0.01, 0.99, 99)
costs = [cost_at(t, y_val, p_val) for t in grid]
best_t = grid[int(np.argmin(costs))]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(grid, costs, color="#2c7fb8")
ax.axvline(best_t, color="#c0392b", ls="--", label=f"cheapest threshold = {best_t:.2f}")
ax.set_xlabel("threshold"); ax.set_ylabel("total cost on validation set"); ax.legend()
plt.tight_layout(); plt.show()

print(f"Theory says a well-calibrated model's best threshold is")
print(f"  COST_FP / (COST_FP + COST_FN) = {COST_FP/(COST_FP+COST_FN):.3f}")
print(f"We found {best_t:.3f} empirically. Close-ish = reasonably calibrated.")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

flag = (p_sk >= best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, flag).ravel()

print(f"On the test set, at threshold {best_t:.2f}:\n")
print(f"  Correctly flagged (true positives) : {tp:4d}")
print(f"  Missed cases      (false negatives): {fn:4d}  <- the expensive ones")
print(f"  False alarms      (false positives): {fp:4d}")
print(f"  Correctly cleared (true negatives) : {tn:4d}")
print()
print(f"  Recall    (of all real cases, how many did we catch?) : {tp/(tp+fn):.1%}")
print(f"  Precision (of everyone we flagged, how many were real?): {tp/(tp+fp):.1%}")
print()
print(f"  ROC-AUC : {roc_auc_score(y_test, p_sk):.4f}")
print(f"  PR-AUC  : {average_precision_score(y_test, p_sk):.4f}  (vs {y_test.mean():.4f} for random guessing)")

In [ ]:
# Compare against the two naive baselines the assignment asks for.
rows = []
for name, pp in [("flag nobody", np.zeros_like(p_sk)),
                 ("flag everyone", np.ones_like(p_sk)),
                 ("our model", p_sk)]:
    f = (pp >= best_t).astype(int) if name == "our model" else (pp > 0.5).astype(int)
    c = cost_at(0.5, y_test, pp) if name != "our model" else cost_at(best_t, y_test, pp)
    rows.append({"strategy": name,
                 "caught": int(((f == 1) & (y_test == 1)).sum()),
                 "false alarms": int(((f == 1) & (y_test == 0)).sum()),
                 "total cost": c})
pd.DataFrame(rows)

### Which features mattered?Because everything was standardized, coefficients are directly comparable: each is"change in log-odds per one standard deviation of this feature".

In [ ]:
coefs = pd.Series(sk.coef_[0], index=feat_names).sort_values()
top = pd.concat([coefs.head(12), coefs.tail(12)])

fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(top.index, top.values, color=["#2c7fb8" if v < 0 else "#c0392b" for v in top.values])
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("coefficient (red = raises risk, blue = lowers it)")
plt.tight_layout(); plt.show()

## Part 3 — Your turnEverything above is the "before" picture. The 30% of your grade that is feature engineeringhappens by changing things and measuring whether they helped.**The loop:** change one thing -> rerun -> write down what PR-AUC did -> keep or discard.Use the cell below as a scratchpad. When something works, move it into`engineer_features()` in `common.py` so it becomes part of the real pipeline.

In [ ]:
def try_features(df):
    # Edit this, then run the cell. Returns a modified copy of the raw dataframe.
    df = df.copy()

    # --- IDEA 1: treat GeneralHealth as an ordered number instead of one-hot ---
    # order = ["Excellent", "Very good", "Good", "Fair", "Poor"]
    # df["GeneralHealth_ord"] = df["GeneralHealth"].map({v: i for i, v in enumerate(order)})
    # df = df.drop(columns=["GeneralHealth"])

    # --- IDEA 2: drop columns you decided are leakage ---
    # df = df.drop(columns=["HadAngina", "ChestScan"])

    # --- IDEA 3: combine the disability columns into a single count ---
    # dis = ["DifficultyWalking", "DifficultyErrands", "DifficultyDressingBathing",
    #        "DifficultyConcentrating", "DeafOrHardOfHearing", "BlindOrVisionDifficulty"]
    # df["n_difficulties"] = (df[dis] == "Yes").sum(axis=1)
    # df = df.drop(columns=dis)

    return df


def evaluate(df, label):
    # Run the whole pipeline on a modified dataframe and report validation PR-AUC.
    yy = df[TARGET].eq("Yes").astype(int).to_numpy()
    XX = df.drop(columns=[TARGET])
    Xtv, Xt_, ytv, yt_ = train_test_split(XX, yy, test_size=0.2, stratify=yy, random_state=SEED)
    Xtr_, Xva_, ytr_, yva_ = train_test_split(Xtv, ytv, test_size=0.25, stratify=ytv, random_state=SEED)

    num = [c for c in Xtr_.columns if pd.api.types.is_numeric_dtype(Xtr_[c])]
    cat = [c for c in Xtr_.columns if c not in num]
    p = Pipeline([("encode", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), num),
            ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore",
                                                       sparse_output=False))]), cat),
         ], verbose_feature_names_out=False)), ("scale", StandardScaler())])

    A, B = p.fit_transform(Xtr_), p.transform(Xva_)
    m = LogisticRegression(C=0.01, max_iter=10_000).fit(A, ytr_)
    pr = average_precision_score(yva_, m.predict_proba(B)[:, 1])
    print(f"{label:35s} features={A.shape[1]:4d}   validation PR-AUC = {pr:.4f}")
    return pr

base = evaluate(raw, "baseline (nothing changed)")
mine = evaluate(try_features(raw), "with my changes")
print(f"\nChange: {mine - base:+.4f}")

### Keep a logMake a table like this in REPORT.md as you go. The honest entries — the ones where PR-AUCwent **down** — are worth as much as the wins. The README explicitly asks you to be honestabout what did not work.| # | What I changed | Why I thought it would help | Val PR-AUC | Kept? ||---|---|---|---|---|| 0 | baseline, nothing | — | 0.4211 | — || 1 | | | | || 2 | | | | |### When something worksMove it into `engineer_features()` in `common.py`, then from a terminal in the repo folder:```uv run python main.py train```That regenerates the real saved models, and the Gradio dashboard will pick them up.